In [ ]:
# Data collection 
import pandas as pd
import requests
from bs4 import BeautifulSoup

def collect_data():
    # Example: Collecting sample movie reviews (Simulated scraping for the demo)
    # In your real task, you can use a Kaggle CSV or a specific URL
    data = {
        'review_id': [1, 2, 3, 4, 5],
        'user_name': ['Youssef', 'Ahmed', 'Marwan', 'None', 'Yasser'],
        'text_content': [
            "The movie was absolutely amazing! 10/10",
            "Terrible experience, would not recommend.",
            "It was okay, but a bit too long.",
            "!!! ERROR !!! No content here.",
            "Loved the acting but hated the plot."
        ],
        'rating': [10, 2, 5, 0, 7],
        'source': ['IMDb', 'IMDb', 'Letterboxd', 'IMDb', 'RottenTomatoes']
    }
    df = pd.DataFrame(data)
    df.to_csv('movie_reviews_dataset.csv', index=False)
    return df

df = collect_data()
print("Dataset Created!")

Dataset Created!


In [4]:
# Data Preparation & Cleaning
def clean_data(df):
    # 1. Handle Missing Values
    df['user_name'] = df['user_name'].replace('None', 'Anonymous')
    
    # 2. Text Cleaning: Lowercase and stripping white spaces
    df['text_content'] = df['text_content'].str.strip().str.lower()
    
    # 3. Anomaly Discovery: Removing rows with suspicious text
    df = df[~df['text_content'].str.contains("error")]
    
    # 4. Preparing labels based on rating
    df['sentiment'] = df['rating'].apply(lambda x: 'Positive' if x >= 7 else ('Neutral' if x >= 5 else 'Negative'))
    
    return df

df_cleaned = clean_data(df)

C:\Users\User\AppData\Local\Temp\ipykernel_5108\4066316432.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = df['rating'].apply(lambda x: 'Positive' if x >= 7 else ('Neutral' if x >= 5 else 'Negative'))


In [ ]:
class TrustGuardValidator:
    def __init__(self, schema_config):
        self.schema = schema_config

    def validate(self, data_row):
        """
        Validates a single row of text data against the schema.
        """
        errors = []
        text = data_row.get('text_content', '')
        
        # Criteria 1: Minimum Length
        if len(text) < self.schema.get('min_length', 1):
            errors.append("Text too short")
            
        # Criteria 2: Mandatory Keywords (e.g., must contain a rating-related word)
        if self.schema.get('check_keywords'):
            keywords = ['movie', 'acting', 'plot', 'recommend', 'experience']
            if not any(word in text for word in keywords):
                errors.append("Missing context keywords")
        
        # Criteria 3: Rating Range
        rating = data_row.get('rating', 0)
        if not (self.schema.get('min_rating') <= rating <= self.schema.get('max_rating')):
            errors.append(f"Rating {rating} out of range")

        return {"is_valid": len(errors) == 0, "errors": errors}

# --- Usage of the Validator ---
my_schema = {
    'min_length': 10,
    'check_keywords': True,
    'min_rating': 1,
    'max_rating': 10
}

validator = TrustGuardValidator(my_schema)

# Apply validation
df_cleaned['validation_results'] = df_cleaned.apply(lambda row: validator.validate(row), axis=1)